# Bài tập 07 - Đánh giá mô hình

Sinh viên: **Lê Quyết Tiến** - MSSV: **2474802010386**.

Notebook chạy từ thư mục gốc `07. Accuracy_Metrics`. Trên Colab, hãy mount Google Drive, `%cd` đến thư mục này rồi chọn **Runtime > Run all**. Giữ nguyên hai tệp CSV trong `data/`.


## Bài 1


In [1]:
# -*- coding: utf-8 -*-
"""Bài 1: tự viết hàm đánh giá chỉ bằng numpy."""

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def chia_an_toan(tu_so, mau_so):
    return tu_so / mau_so if mau_so != 0 else 0.0


def danh_gia(y_that, y_doan):
    y_that = np.asarray(y_that)
    y_doan = np.asarray(y_doan)
    tp = np.sum((y_that == 1) & (y_doan == 1))
    fp = np.sum((y_that == 0) & (y_doan == 1))
    fn = np.sum((y_that == 1) & (y_doan == 0))
    tn = np.sum((y_that == 0) & (y_doan == 0))
    accuracy = chia_an_toan(tp + tn, tp + fp + fn + tn)
    precision = chia_an_toan(tp, tp + fp)
    recall = chia_an_toan(tp, tp + fn)
    f1 = chia_an_toan(2 * precision * recall, precision + recall)
    return accuracy, precision, recall, f1


df = pd.read_csv("data/sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
mo_hinh = {
    "Logistic": make_pipeline(StandardScaler(), LogisticRegression()),
    "Mo hinh luoi": DummyClassifier(strategy="most_frequent"),
}
for ten, mh in mo_hinh.items():
    du = mh.fit(X_tr, y_tr).predict(X_te)
    tu_tinh = danh_gia(y_te, du)
    thu_vien = (
        accuracy_score(y_te, du),
        precision_score(y_te, du, zero_division=0),
        recall_score(y_te, du, zero_division=0),
        f1_score(y_te, du, zero_division=0),
    )
    print(ten)
    print("  Tu tinh :", tuple(round(float(x), 4) for x in tu_tinh))
    print("  Thu vien:", tuple(round(x, 4) for x in thu_vien))

print("\nNhan xet va giai thich:")
print("- Mo hinh luoi khong bao mot mau duong nao, nen TP+FP=0 va precision co phep chia cho 0.")
print("- Ve mat toan hoc precision khi do khong xac dinh. Ham nay quy uoc tra 0.0, giong")
print("  scikit-learn khi dat zero_division=0, de chuong trinh khong loi va y nghia ro rang.")
print("- Accuracy cua mo hinh luoi cao do lop 0 chiem da so, nhung recall lop bo hoc bang 0.")


Logistic
  Tu tinh : (0.9467, 0.85, 0.5667, 0.68)
  Thu vien: (0.9467, 0.85, 0.5667, 0.68)
Mo hinh luoi
  Tu tinh : (0.9, 0.0, 0.0, 0.0)
  Thu vien: (0.9, 0.0, 0.0, 0.0)

Nhan xet va giai thich:
- Mo hinh luoi khong bao mot mau duong nao, nen TP+FP=0 va precision co phep chia cho 0.
- Ve mat toan hoc precision khi do khong xac dinh. Ham nay quy uoc tra 0.0, giong
  scikit-learn khi dat zero_division=0, de chuong trinh khong loi va y nghia ro rang.
- Accuracy cua mo hinh luoi cao do lop 0 chiem da so, nhung recall lop bo hoc bang 0.


## Bài 2


In [2]:
# -*- coding: utf-8 -*-
"""Bài 2: classification_report cho SVM nhận dạng 10 chữ số."""

import numpy as np
from sklearn.datasets import load_digits
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
svm = SVC(kernel="rbf").fit(X_tr, y_tr)
du = svm.predict(X_te)
print(classification_report(y_te, du, digits=4))
bao_cao = classification_report(y_te, du, output_dict=True, zero_division=0)
recall = {int(k): v["recall"] for k, v in bao_cao.items() if k.isdigit()}
precision = {int(k): v["precision"] for k, v in bao_cao.items() if k.isdigit()}
r_min = min(recall.values())
p_min = min(precision.values())
so_recall_thap = [k for k, v in recall.items() if np.isclose(v, r_min)]
so_precision_thap = [k for k, v in precision.items() if np.isclose(v, p_min)]
print("Chu so co recall thap nhat:", so_recall_thap, f"({r_min:.4f})")
print("Chu so co precision thap nhat:", so_precision_thap, f"({p_min:.4f})")
print("\nNhan xet:")
print("- Cac chu so nay co net viet gan voi lop khac, nen de bi bo sot hoac nhan nham.")
print("- Moi lop co khoang 44-46 anh trong tap kiem tra. Vi support gan bang nhau, macro avg")
print("  va weighted avg gan bang nhau; khong co lop dong nao chi phoi trung binh co trong so.")


              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        45
           1     0.9787    1.0000    0.9892        46
           2     1.0000    1.0000    1.0000        44
           3     0.9787    1.0000    0.9892        46
           4     1.0000    0.9778    0.9888        45
           5     1.0000    1.0000    1.0000        46
           6     1.0000    1.0000    1.0000        45
           7     1.0000    0.9778    0.9888        45
           8     0.9767    0.9767    0.9767        43
           9     0.9778    0.9778    0.9778        45

    accuracy                         0.9911       450
   macro avg     0.9912    0.9910    0.9911       450
weighted avg     0.9912    0.9911    0.9911       450

Chu so co recall thap nhat: [8] (0.9767)
Chu so co precision thap nhat: [8] (0.9767)

Nhan xet:
- Cac chu so nay co net viet gan voi lop khac, nen de bi bo sot hoac nhan nham.
- Moi lop co khoang 44-46 anh trong tap kiem tra. Vi support gan 

## Bài 3


In [3]:
# -*- coding: utf-8 -*-
"""Bài 3: chọn ngưỡng lớn nhất để recall kiểm định chéo đạt ít nhất 0.8."""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
lr = make_pipeline(StandardScaler(), LogisticRegression())
xs_hoc = cross_val_predict(lr, X_tr, y_tr, cv=5, method="predict_proba")[:, 1]
cac_nguong = np.round(np.arange(0.01, 1.00, 0.01), 2)
hop_le = []
for t in cac_nguong:
    du_hoc = (xs_hoc >= t).astype(int)
    if recall_score(y_tr, du_hoc) >= 0.8:
        hop_le.append(t)
nguong = max(hop_le)
recall_cv = recall_score(y_tr, (xs_hoc >= nguong).astype(int))

lr.fit(X_tr, y_tr)
xs_te = lr.predict_proba(X_te)[:, 1]
du_te = (xs_te >= nguong).astype(int)
print(f"Nguong lon nhat co recall CV >= 0.8: {nguong:.2f}")
print(f"Recall tren du doan cheo tap hoc: {recall_cv:.4f}")
print(f"Tap kiem tra - recall:    {recall_score(y_te, du_te):.4f}")
print(f"Tap kiem tra - precision: {precision_score(y_te, du_te):.4f}")
print(f"So ban bi bao: {du_te.sum()} tren {len(du_te)}")

print("\nNhan xet va giai thich:")
print("- Nguong duoc chon chi tu xac suat cross_val_predict tren tap hoc. Moi xac suat den tu")
print("  mot mo hinh chua tung thay mau do, nen khong nhin dap an cua tap kiem tra.")
print("- Chon nguong lon nhat van dat recall >=0.8 giup han che bao dong nham trong cac nguong")
print("  thoa yeu cau. Tap kiem tra chi dung mot lan sau khi nguong da duoc khoa.")
print("- Recall tap kiem tra co the khac 0.8 do sai khac lay mau; yeu cau tren tap hoc khong bao")
print("  dam chinh xac cung mot con so tren du lieu moi.")


Nguong lon nhat co recall CV >= 0.8: 0.17
Recall tren du doan cheo tap hoc: 0.8090
Tap kiem tra - recall:    0.8000
Tap kiem tra - precision: 0.5106
So ban bi bao: 47 tren 300

Nhan xet va giai thich:
- Nguong duoc chon chi tu xac suat cross_val_predict tren tap hoc. Moi xac suat den tu
  mot mo hinh chua tung thay mau do, nen khong nhin dap an cua tap kiem tra.
- Chon nguong lon nhat van dat recall >=0.8 giup han che bao dong nham trong cac nguong
  thoa yeu cau. Tap kiem tra chi dung mot lan sau khi nguong da duoc khoa.
- Recall tap kiem tra co the khac 0.8 do sai khac lay mau; yeu cau tren tap hoc khong bao
  dam chinh xac cung mot con so tren du lieu moi.


## Bài 4


In [4]:
# -*- coding: utf-8 -*-
"""Bài 4: tự vẽ đường ROC của logistic và rừng ngẫu nhiên."""

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
mo_hinh = {
    "Logistic": make_pipeline(StandardScaler(), LogisticRegression()),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=0),
}

fig, ax = plt.subplots(figsize=(7, 5))
duong = {}
for ten, mh in mo_hinh.items():
    p = mh.fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, p)
    auc = roc_auc_score(y_te, p)
    duong[ten] = (fpr, tpr, auc)
    ax.plot(fpr, tpr, linewidth=2, label=f"{ten}, AUC={auc:.4f}")
ax.plot([0, 1], [0, 1], "--", color="gray", label="Doan ngau nhien")
ax.set_xlabel("Ty le bao dong nham (FPR)")
ax.set_ylabel("Recall (TPR)")
ax.set_title("Duong ROC")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/roc.png", dpi=160, bbox_inches="tight")
if "agg" not in plt.get_backend().lower():
    plt.show()
else:
    plt.close(fig)

luoi = np.linspace(0, 1, 1001)
tpr_lr = np.interp(luoi, duong["Logistic"][0], duong["Logistic"][1])
tpr_rf = np.interp(luoi, duong["Random Forest"][0], duong["Random Forest"][1])
chenh = tpr_lr - tpr_rf
cat_nhau = np.any(chenh[:-1] * chenh[1:] < 0)
print(f"AUC Logistic:      {duong['Logistic'][2]:.4f}")
print(f"AUC Random Forest: {duong['Random Forest'][2]:.4f}")
print("Hai duong co doan doi thu tu/cat nhau:", bool(cat_nhau))
print("\nNhan xet:")
print("- Duong co AUC cao hon nhin chung om sat goc tren ben trai hon.")
print("- Hai duong co the cat nhau o mot so vung nguong, vi mot mo hinh co the tot hon tai FPR")
print("  thap nhung kem hon tai FPR cao. AUC tom tat toan bo duong chu khong mo ta moi diem.")
print("- Da luu hinh tai figures/roc.png.")


AUC Logistic:      0.9449
AUC Random Forest: 0.9249
Hai duong co doan doi thu tu/cat nhau: True

Nhan xet:
- Duong co AUC cao hon nhin chung om sat goc tren ben trai hon.
- Hai duong co the cat nhau o mot so vung nguong, vi mot mo hinh co the tot hon tai FPR
  thap nhung kem hon tai FPR cao. AUC tom tat toan bo duong chu khong mo ta moi diem.
- Da luu hinh tai figures/roc.png.


## Bài 5


In [5]:
# -*- coding: utf-8 -*-
"""Bài 5: cân bằng lớp cho Random Forest và SVC."""

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

df = pd.read_csv("data/sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
mo_hinh = {
    "RF binh thuong": RandomForestClassifier(n_estimators=200, random_state=0),
    "RF can bang": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=0),
    "SVC binh thuong": make_pipeline(StandardScaler(), SVC()),
    "SVC can bang": make_pipeline(StandardScaler(), SVC(class_weight="balanced")),
}
ket_qua = {}
print("Mo hinh          precision recall F1")
for ten, mh in mo_hinh.items():
    du = mh.fit(X_tr, y_tr).predict(X_te)
    p = precision_score(y_te, du)
    r = recall_score(y_te, du)
    f1 = f1_score(y_te, du)
    ket_qua[ten] = (p, r, f1)
    print(f"{ten:16s} {p:.4f}    {r:.4f} {f1:.4f}")

doi_rf = sum(abs(a-b) for a, b in zip(ket_qua["RF binh thuong"], ket_qua["RF can bang"]))
doi_svc = sum(abs(a-b) for a, b in zip(ket_qua["SVC binh thuong"], ket_qua["SVC can bang"]))
print("\nNhan xet:")
print(f"- Tong thay doi ba thuoc do: Random Forest={doi_rf:.4f}, SVC={doi_svc:.4f}.")
print(f"- {'SVC' if doi_svc > doi_rf else 'Random Forest'} thay doi nhieu hon theo tong chenh lech nay.")
print("- class_weight='balanced' phat nang loi o lop hiem, thuong lam recall tang nhung precision")
print("  co the giam vi mo hinh bao nhieu mau duong hon.")


Mo hinh          precision recall F1
RF binh thuong   0.7727    0.5667 0.6538
RF can bang      0.7500    0.4000 0.5217
SVC binh thuong  0.8824    0.5000 0.6383
SVC can bang     0.4464    0.8333 0.5814

Nhan xet:
- Tong thay doi ba thuoc do: Random Forest=0.3215, SVC=0.8262.
- SVC thay doi nhieu hon theo tong chenh lech nay.
- class_weight='balanced' phat nang loi o lop hiem, thuong lam recall tang nhung precision
  co the giam vi mo hinh bao nhieu mau duong hon.


## Bài 6


In [6]:
# -*- coding: utf-8 -*-
"""Bài 6: MAE, RMSE và R2 trước/sau một dự đoán hỏng."""

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/gia_nha.csv")
X = df[["dien_tich", "so_phong", "tuoi_nha"]]
y = df["gia"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17
)
mh = LinearRegression().fit(X_tr, y_tr)
du_goc = mh.predict(X_te)


def tinh_chi_so(y_that, y_doan):
    return (
        mean_absolute_error(y_that, y_doan),
        np.sqrt(mean_squared_error(y_that, y_doan)),
        r2_score(y_that, y_doan),
    )


goc = tinh_chi_so(y_te, du_goc)
du_hong = du_goc.copy()
du_hong[0] += 5.0
hong = tinh_chi_so(y_te, du_hong)
print("                 MAE      RMSE       R2")
print(f"Du doan goc     {goc[0]:.4f}   {goc[1]:.4f}   {goc[2]:.4f}")
print(f"Mot du doan +5  {hong[0]:.4f}   {hong[1]:.4f}   {hong[2]:.4f}")
print("\nMuc thay doi tuyet doi:")
print(f" MAE:  {abs(hong[0]-goc[0]):.4f}")
print(f" RMSE: {abs(hong[1]-goc[1]):.4f}")
print(f" R2:   {abs(hong[2]-goc[2]):.4f}")
print("\nNhan xet:")
print("- RMSE tang manh hon MAE theo don vi gia, vi RMSE binh phuong tung sai so truoc khi lay")
print("  trung binh. Sai so lon 5 ty dong tao thanh 25 trong tong binh phuong sai so.")
print("- MAE chi cong do lon sai so nen moi diem anh huong tuyen tinh. R2 giam vi tong sai so")
print("  binh phuong tang, cho thay mo hinh giai thich du lieu kem hon sau lan doan hong.")


                 MAE      RMSE       R2
Du doan goc     0.2185   0.2700   0.9759
Mot du doan +5  0.5519   1.4433   0.3110

Muc thay doi tuyet doi:
 MAE:  0.3333
 RMSE: 1.1733
 R2:   0.6649

Nhan xet:
- RMSE tang manh hon MAE theo don vi gia, vi RMSE binh phuong tung sai so truoc khi lay
  trung binh. Sai so lon 5 ty dong tao thanh 25 trong tong binh phuong sai so.
- MAE chi cong do lon sai so nen moi diem anh huong tuyen tinh. R2 giam vi tong sai so
  binh phuong tang, cho thay mo hinh giai thich du lieu kem hon sau lan doan hong.
